# Aletheia's Quest submission
Entry point: a stable `harness` runs the method module selected by `ALETHEIA_METHOD` (default set in the next cell; on the leaderboard no env is set, so the default ships). The last cell guarantees a valid `submission.csv` is always written.

## Method report — `rhadamanthys_dyad_anchor2h` (tag black)

This notebook runs exactly one detector module — **`rhadamanthys_dyad_anchor2h`**, selected by
`ALETHEIA_METHOD` below. The same detection logic runs for every model organism and every dataset:
there is no per-organism branching anywhere on the call path.

It is a black-box method: the trusted judge `Qwen/Qwen3.5-9B` is the only model read. The suspect's
logprobs and activations are never touched, and no trained weights are loaded.

The report is kept beside this notebook rather than inside it, so that it cannot drift from the
method that was actually fired:

| document | contents |
|---|---|
| `PROVENANCE.md` | what the four judge passes do, the board result, and the exact source of every file |
| `RUN.md` | how to run this bundle, locally or through NDIF |

> **Note on this cell (added for the public release).** In the fired notebook, cells 1 and 2
> carried a method report for a *different, white-box* submission (`rhadamanthys_m53`), left behind
> by an earlier revision of this template — including a board headline that is not this entry's and
> pointers to `submission/training/` and `pm22.npz`, neither of which exists in this bundle. The
> code cells below are untouched and are what actually ran. See `PROVENANCE.md`.

## Training reproducibility

`rhadamanthys_dyad_anchor2h` loads **no trained artifact**. It is prompt-programmed judges only:
four passes over a stock `Qwen/Qwen3.5-9B`, with fixed prompt texts that live in the bundled
`methods/` modules. There is nothing to retrain and no checkpoint to verify.

The next cell is the competition's generic retrain hook. It is a no-op during scoring, and for this
method it is a no-op entirely — it looks for a `submission/training/` directory that this bundle
does not ship, because this method needs none.

In [ ]:
# --- Training reproducibility (competition rule: training contained in the submission and
# reproducible when organizers re-run it). This cell is a NO-OP during scoring. ---
# Organizers: to reproduce the only trained artifact (submission/pm22.npz, the white-hedge
# directions), set  ALETHEIA_RETRAIN=1  (deterministic, numpy-only, asserts an exact match to the
# shipped artifact) or  ALETHEIA_RETRAIN=scratch  (also re-derives the qwen leg via NDIF; needs
# .env + network). Full chain + the activation-extraction step: submission/training/PROVENANCE.md.
import os as _os, subprocess as _sp, sys as _sys
_retrain = _os.environ.get("ALETHEIA_RETRAIN", "")
if _retrain and _os.path.isdir("submission/training"):
    _cmd = [_sys.executable, "submission/training/rebuild_pm22.py"]
    if _retrain == "scratch":
        _cmd.append("--from-scratch")            # NDIF re-derivation may not be bit-identical
    else:
        _cmd += ["--verify", "submission/pm22.npz"]  # deterministic: assert exact reproduction
    print(f"[submission] ALETHEIA_RETRAIN={_retrain} -> rebuilding pm22.npz from submission/training/")
    _sp.run(_cmd, check=True)
else:
    print("[submission] scoring run: pm22.npz used as shipped. "
          "Set ALETHEIA_RETRAIN=1 to reproduce it (see submission/training/PROVENANCE.md).")

In [ ]:
import os, sys
sys.path.insert(0, "submission")
import util

DATASET_NAME = os.environ["DATASET_NAME"]
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None
METHOD = os.environ.get("ALETHEIA_METHOD", "rhadamanthys_dyad_anchor2h")   # default shipped on the leaderboard

In [ ]:
# On ANY failure still write a valid submission.csv (neutral 0.5) so the run never hard-fails.
import numpy as np, traceback
try:
    import harness, methods
    harness.run_submission(methods.load(METHOD), DATASET_NAME, limit=LIMIT, batch_size=8, max_len=256, remote=True)
except Exception as e:
    traceback.print_exc()
    print(f"[submission] harness failed ({type(e).__name__}: {e}) -> writing 0.5 fallback", flush=True)
    ex = util.load_examples(DATASET_NAME)
    n = len(ex) if LIMIT is None else min(LIMIT, len(ex))
    util.write_submission(list(ex["index"])[:n], np.full(n, 0.5))

In [ ]:
# Ensure submission.csv exists no matter what.
import os as _os, numpy as _np
if not _os.path.exists("submission.csv"):
    _ex = util.load_examples(DATASET_NAME)
    _n = len(_ex) if LIMIT is None else min(LIMIT, len(_ex))
    util.write_submission(list(_ex["index"])[:_n], _np.full(_n, 0.5))
    print("[submission] wrote final 0.5 safety net")